In [1]:
import os
import sys
sys.path.append('../../')
import jax
import optax
from typing import List

from evosax.problems import CNN, TorchVisionProblem, identity_output_fn
from evosax.algorithms import algorithms
from evosax.core.fitness_shaping import standardize_fitness_shaping_fn

from tqdm import tqdm
from utils.problem_utils import get_problem_settings
from experiment.experiment import Experiment
from utils.problem_utils import get_problem_name
import torch
# from experiment.run_experiments import run_experiment_permutations

# jax.config.update('jax_default_matmul_precision', 'tensorfloat32')
# os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'  # 0 = all logs, 1 = filter INFO, 2 = filter WARNING, 3 = filter ERROR
# os.environ['XLA_PYTHON_CLIENT_PREALLOCATE'] = 'false'
# os.environ['XLA_FLAGS'] = '--xla_gpu_strict_conv_algorithm_picker=false --xla_gpu_autotune_level=1'

In [2]:
import evosax
evosax.algorithms.distribution_based.cma_es.CMA_ES?

Init signature:
evosax.algorithms.distribution_based.cma_es.CMA_ES(
    population_size: int,
    solution: Any,
    fitness_shaping_fn: collections.abc.Callable = <function weights_fitness_shaping_fn at 0x7fb2a8815d00>,
    metrics_fn: collections.abc.Callable = <function metrics_fn at 0x7fb2a882dee0>,
)
Docstring:      Covariance Matrix Adaptation Evolution Strategy (CMA-ES).
Init docstring: Initialize CMA-ES.
File:           ~/.conda/envs/phdEnv/lib/python3.11/site-packages/evosax/algorithms/distribution_based/cma_es.py
Type:           type
Subclasses:     MA_ES, Rm_ES, Sep_CMA_ES, SV_CMA_ES

In [3]:
es_dict = {
#     "PGPE": {
#         "optimizer": optax.adam(learning_rate=0.02),
#     },
#     "ASEBO": {
#         "optimizer": optax.adam(learning_rate=0.01),
#         "fitness_shaping_fn": standardize_fitness_shaping_fn
#     },
#     "LES": {
#         "optimizer": optax.adam(learning_rate=0.01)
#     },
#     "Open_ES": {    
#         "optimizer": optax.adam(learning_rate=0.05)
#     },
    
    # "SNES": {},
    # "Sep_CMA_ES": {},
    "CMA_ES": {},
    # "LES": {},
    # "DES": {},
    # "EvoTF_ES": {},
}

# take only the es_algorithms we insert as arg.
running_es = es_dict

In [4]:
from evosax.problems import Problem

def run_experiment_permutations(problems: List[Problem], es_dict: dict, num_generations: int, population_size: int,
                                result_dir: str, run_again_if_exist: bool = False,
                                seeds: list[int] = list(range(0, 5))):
    for problem in problems:
        for es_name in tqdm(es_dict, desc="Running ES algorithms"):
            try:
                for seed in seeds:
                    key = jax.random.key(seed)
                    key, subkey = jax.random.split(key)
                    # solution = problem.sample(subkey)
                    solution = jax.tree_map(lambda x: x * 0.01, problem.sample(key))
                    es_algorithm = algorithms[es_name](population_size=population_size,
                                                       solution=solution,
                                                       **es_dict[es_name])
                    experiment = Experiment(problem=problem,
                                            algorithm=es_algorithm,
                                            results_dir_path=result_dir,
                                            seed=seed, log_period=LOG_PERIOD, eval_batch_size=EVAL_BATCH_SIZE)
     
                    

                    if not experiment.has_run() or run_again_if_exist:
                        print(f"running the experiment ... [{experiment.get_experiment_path_file()}]")
                        experiment.run(num_generations=num_generations)
                    else:
                        print(f"there is experiment results. [{experiment.get_experiment_path_file()}]")
            except Exception as e:
                print(e, es_name, get_problem_name(problem))
                raise e

In [6]:
NUM_GENERATIONS = 100
POPULATION_SIZE = 64
EVAL_BATCH_SIZE = 16
LOG_PERIOD = 2
SEEDS = [0]
RESULT_DIR = "../../results"
# PROBLEMS_TORCH_VISION = ["MNIST", "FashionMNIST", "CIFAR10", "SVHN"]
PROBLEMS_TORCH_VISION = ["MNIST"]

In [7]:
for task_name in tqdm(PROBLEMS_TORCH_VISION, desc="Loading Problems .."):
    try:
        problem = TorchVisionProblem(task_name=task_name,
                          network=CNN(
                              num_filters=[4],
                              kernel_sizes=[(5, 5)],
                              strides=[(1, 1)],
                              mlp_layer_sizes=[5],
                              activation=jax.nn.tanh,
                              output_fn=identity_output_fn,
                          ),
                          batch_size=64)
        print("Successfully loaded:", task_name)
        run_experiment_permutations(problems=[problem],
                                    es_dict=running_es,
                                    num_generations=NUM_GENERATIONS,
                                    population_size=POPULATION_SIZE,
                                    result_dir=RESULT_DIR, 
                                    run_again_if_exist=True,
                                    seeds=SEEDS)
    except Exception as e:
        print("Failed to load: " + task_name,'\nREASON:', e)
        raise e

Loading Problems ..:   0%|          | 0/1 [00:00<?, ?it/s]

Successfully loaded: MNIST



Running ES algorithms:   0%|          | 0/1 [00:00<?, ?it/s]/tmp/ipykernel_1185217/1204499789.py:13: DeprecationWarning: jax.tree_map is deprecated: use jax.tree.map (jax v0.4.25 or newer) or jax.tree_util.tree_map (any JAX version).
  solution = jax.tree_map(lambda x: x * 0.01, problem.sample(key))


running the experiment ... [../../results/TorchVisionProblem/MNIST/CMA_ES/0]
Generation 002 | Mean fitness (Test): nan | Mean Accuracy (Test): 0.12
Generation 004 | Mean fitness (Test): nan | Mean Accuracy (Test): 0.16
Generation 006 | Mean fitness (Test): nan | Mean Accuracy (Test): 0.11
Generation 008 | Mean fitness (Test): nan | Mean Accuracy (Test): 0.06
Generation 010 | Mean fitness (Test): nan | Mean Accuracy (Test): 0.06
Generation 012 | Mean fitness (Test): nan | Mean Accuracy (Test): 0.05
Generation 014 | Mean fitness (Test): nan | Mean Accuracy (Test): 0.05
Generation 016 | Mean fitness (Test): nan | Mean Accuracy (Test): 0.05
Generation 018 | Mean fitness (Test): nan | Mean Accuracy (Test): 0.05
Generation 020 | Mean fitness (Test): nan | Mean Accuracy (Test): 0.05
Generation 022 | Mean fitness (Test): nan | Mean Accuracy (Test): 0.06
Generation 024 | Mean fitness (Test): nan | Mean Accuracy (Test): 0.05
Generation 026 | Mean fitness (Test): nan | Mean Accuracy (Test): 0.06


Loading Problems ..:   0%|          | 0/1 [05:36<?, ?it/s]


KeyboardInterrupt: 